# Agentic Literature Review & Learning Path Recommender

**CIS 600 — Applied Agentic AI Systems | Spring 2026**

**Team Members:** Rajnish Sahani, Deven Wagh, Hangye Li, Yonghao Li

**Instructor:** Professor Kumarawadu

---

This notebook demonstrates our multi-agent system that takes a research topic as input and produces:
- A curated set of relevant papers (from arXiv + Semantic Scholar)
- Relevance-scored screening
- Thematic synthesis across papers  
- Research gap identification
- A **recommended reading order** — so a beginner knows exactly which paper to start with

The actual system lives in `.py` files (agents/, controller/, tools/, etc.). This notebook imports and runs that code, then visualizes the results.

## 1. Setup

In [1]:
import os
import sys
import time
import json
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

# make sure the key is actually there
assert os.getenv("GROQ_API_KEY") is not None, "GROQ_API_KEY not found in .env"
print("API key loaded successfully")

API key loaded successfully


## 2. Project Structure

Each agent is its own file. The orchestrator wires them together using LangGraph.

```
Agentic-lit-review/
├── agents/
│   ├── search_agent.py        # generates queries, hits arXiv + Semantic Scholar
│   ├── screening_agent.py     # scores relevance 0-1, filters out irrelevant papers
│   ├── synthesis_agent.py     # summarizes papers, identifies themes
│   └── planning_agent.py      # gaps, research plan, reading order
├── controller/
│   └── orchestrator.py        # LangGraph StateGraph connects everything
├── tools/
│   ├── arxiv_tool.py          # arXiv API wrapper
│   └── semantic_scholar.py    # Semantic Scholar API wrapper
├── state/
│   └── shared_state.py        # TypedDict state schema
├── evaluation/
│   └── metrics.py             # eval metrics + baseline comparison
├── main.py                    # CLI entry point
└── .env                       # API keys (not committed)
```

## 3. State Schema

The shared state flows through every agent. Each agent reads what it needs and writes its outputs back.

In [1]:
from state.shared_state import AgentState, Paper

import inspect
print("--- Paper ---")
print(inspect.getsource(Paper))
print("--- AgentState ---")
print(inspect.getsource(AgentState))

--- Paper ---
class Paper(TypedDict):
    title: str
    authors: List[str]
    abstract: str
    year: int
    url: str
    relevance_score: Optional[float]
    summary: Optional[str]
    themes: Optional[List[str]]

--- AgentState ---
class AgentState(TypedDict):
    # Input
    research_topic: str
    
    # Search Agent outputs
    search_queries: List[str]
    raw_papers: List[Paper]
    
    # Screening Agent outputs
    screened_papers: List[Paper]
    
    # Synthesis Agent outputs
    summaries: List[str]
    themes: List[str]
    
    # Planning Agent outputs
    research_gaps: List[str]
    research_plan: str
    reading_order: List[dict]
    
    # Controller
    iteration: int
    max_iterations: int
    status: str
    messages: List[str]


## 4. Agent Orchestration (LangGraph)

The orchestrator builds a StateGraph with 5 nodes. There's a conditional edge after screening — if we didn't find enough papers, it loops back to search. That's what makes this agentic rather than a static pipeline.

In [1]:
from controller.orchestrator import build_graph
import inspect

print(inspect.getsource(build_graph))

def build_graph():
    """Build and return the LangGraph workflow."""
    workflow = StateGraph(AgentState)
    
    workflow.add_node("search", search_agent)
    workflow.add_node("screen", screening_agent)
    workflow.add_node("synthesize", synthesis_agent)
    workflow.add_node("plan", planning_agent)
    workflow.add_node("increment", increment_iteration)
    
    workflow.set_entry_point("search")
    workflow.add_edge("search", "screen")
    workflow.add_edge("screen", "increment")
    
    workflow.add_conditional_edges(
        "increment",
        should_continue,
        {
            "search_again": "search",
            "end": "synthesize"
        }
    )
    
    workflow.add_edge("synthesize", "plan")
    workflow.add_edge("plan", END)
    
    return workflow.compile()


### Graph Flow

```
                    ┌──────────────────────────────────────┐
                    │                                      │
                    ▼                                      │
    [Search Agent] ──→ [Screening Agent] ──→ [Increment]  │
                                                 │        │
                                        ┌────────┴────────┘
                                        │  (< 3 papers? loop back)
                                        │
                                        ▼
                                  [Synthesis Agent] ──→ [Planning Agent] ──→ END
```

The LLM is initialized **inside** each agent function, not at module level. This means swapping providers is a one-line change per file. We started with Gemini, hit quota limits, switched to Groq in 5 minutes.

## 5. Running the Full Pipeline

In [1]:
from controller.orchestrator import run_literature_review

topic = "zero-shot learning"

print(f"Running literature review for: {topic}")
print("This takes about a minute (rate limit delays between agents)...\n")

start_time = time.time()
result = run_literature_review(topic, max_iterations=2)
elapsed = time.time() - start_time

print(f"\nDone in {elapsed:.1f}s")

Running literature review for: zero-shot learning
This takes about a minute (rate limit delays between agents)...

[Search Agent] Searching for: zero-shot learning
[Search Agent] Generated queries: ['zero shot learning AND deep learning', 'few shot learning OR meta learning', 'zero shot classification AND transfer learning']
[Semantic Scholar] Found 4 papers for: zero shot learning AND deep learning
[Semantic Scholar] Request failed: 429 Client Error:  for url: https://api.semanticscholar.org/graph/v1/paper/search?query=few+shot+learning+OR+meta+learning&limit=5&fields=title%2Cauthors%2Cabstract%2Cyear%2CexternalIds%2CopenAccessPdf
[Semantic Scholar] Request failed: 429 Client Error:  for url: https://api.semanticscholar.org/graph/v1/paper/search?query=zero+shot+classification+AND+transfer+learning&limit=5&fields=title%2Cauthors%2Cabstract%2Cyear%2CexternalIds%2CopenAccessPdf
[Search Agent] Found 17 unique papers (arXiv + Semantic Scholar)
[Orchestrator] Waiting 10s for rate limit...



## 6. Results

### 6.1 Papers Retrieved & Screened

In [1]:
print(f"Topic: {result['research_topic']}")
print(f"Search queries generated: {len(result['search_queries'])}")
print(f"Total papers found: {len(result['raw_papers'])}")
print(f"Papers after screening: {len(result['screened_papers'])}")
print(f"Rejection rate: {(len(result['raw_papers']) - len(result['screened_papers'])) / len(result['raw_papers']):.0%}")

print("\n--- Top 5 Papers by Relevance ---")
for i, p in enumerate(result['screened_papers'][:5], 1):
    print(f"\n{i}. {p['title']} ({p.get('year', '?')})")
    print(f"   Relevance: {p.get('relevance_score', 0):.2f}")
    print(f"   Summary: {p.get('summary', 'N/A')[:150]}...")

Topic: zero-shot learning
Search queries generated: 3
Total papers found: 17
Papers after screening: 11
Rejection rate: 35%

--- Top 5 Papers by Relevance ---

1. Generalized Continual Zero-Shot Learning (2020)
   Relevance: 0.90
   Summary: This paper discusses the concept of generalized continual zero-shot learning, which aims to classify unseen classes by transferring knowledge from seen classes without requiring all traini...

2. Zero-shot and Few-shot Learning with Knowledge Graphs: A Comprehensive Survey (2021)
   Relevance: 0.90
   Summary: This survey paper provides a comprehensive overview of zero-shot and few-shot learning with knowledge graphs, highlighting the importance of these techniques in real-world applications wher...

3. Learning a Deep Embedding Model for Zero-Shot Learning (2016)
   Relevance: 0.90
   Summary: This paper proposes a deep embedding model for zero-shot learning, which learns a joint embedding space for textual and visual representations of object cla

### 6.2 Themes Identified

In [1]:
print("Themes found across all screened papers:\n")
for i, theme in enumerate(result['themes'], 1):
    print(f"  {i}. {theme}")

Themes found across all screened papers:

  1. Zero-Shot Learning
  2. Few-Shot Learning
  3. Meta-Learning
  4. Deep Learning
  5. Knowledge Transfer


### 6.3 Research Gaps

In [1]:
print("Identified gaps in the literature:\n")
for i, gap in enumerate(result['research_gaps'], 1):
    print(f"  {i}. {gap}")

Identified gaps in the literature:

  1. Investigating the effectiveness of zero-shot learning in real-world applications with limited labeled data
  2. Developing more efficient and adaptable meta-learning approaches for few-shot learning
  3. Exploring the potential of knowledge graphs in enhancing zero-shot learning and knowledge transfer


### 6.4 Research Plan

In [1]:
print(result['research_plan'])

The research plan aims to investigate the effectiveness of zero-shot learning in real-world applications with limited labeled data. The proposed questions are:
- Can zero-shot learning be effectively applied to real-world applications with limited labeled data?
- How can meta-learning approaches be improved to enhance few-shot learning?
- What is the role of knowledge graphs in enhancing zero-shot learning and knowledge transfer?

The methodology will involve:
- Conducting a comprehensive review of existing literature on zero-shot learning, few-shot learning, and meta-learning
- Developing and evaluating new meta-learning approaches for few-shot learning
- Investigating the use of knowledge graphs in enhancing zero-shot learning and knowledge transfer
- Conducting experiments on real-world datasets to evaluate the effectiveness of zero-shot learning in limited labeled data scenarios

The evaluation strategy will involve:
- Comparing the performance of different meta-learning approaches

### 6.5 Recommended Reading Order (Learning Path)

This is the main feature — not just finding papers but telling you **which order to read them in**. The LLM considers concept dependencies: surveys first, foundational work next, then specialized extensions.

In [1]:
reading_order = result.get('reading_order', [])

if reading_order:
    print(f"Recommended reading sequence ({len(reading_order)} papers):\n")
    for entry in reading_order:
        pos = entry.get('position', '?')
        title = entry.get('title', 'Unknown')
        reason = entry.get('reason', '')
        print(f"  [{pos}] {title}")
        print(f"      -> {reason}")
        print()
else:
    print("No reading order generated")

Recommended reading sequence (11 papers):

  [1] Zero-shot and Few-shot Learning with Knowledge Graphs: A Comprehensive Survey
      -> This survey paper provides a comprehensive overview of zero-shot and few-shot learning with knowledge graphs, making it a foundational paper to start with

  [2] Learning a Deep Embedding Model for Zero-Shot Learning
      -> This paper proposes a deep embedding model for zero-shot learning, which is a key concept in the field and provides a basis for understanding more advanced techniques

  [3] Generalized Continual Zero-Shot Learning
      -> This paper discusses the concept of generalized continual zero-shot learning, which is a more advanced topic that builds on the foundational concepts introduced in the first two papers

  [4] Few-shot Learning with Meta Metric Learners
      -> This paper introduces a meta metric learner for few-shot learning, which is a key concept in meta-learning and provides a basis for understanding more advanced technique

## 7. Visualizations

### 7.1 Relevance Score Distribution

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

scores = [p.get('relevance_score', 0) for p in result['screened_papers']]
titles = [p['title'][:40] + '...' if len(p['title']) > 40 else p['title'] 
          for p in result['screened_papers']]

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2ecc71' if s >= 0.8 else '#f39c12' for s in scores]
ax.barh(range(len(scores)), scores, color=colors)
ax.set_yticks(range(len(titles)))
ax.set_yticklabels(titles, fontsize=8)
ax.set_xlabel('Relevance Score')
ax.set_title('Paper Relevance Scores (after screening)')
ax.set_xlim(0, 1.1)
ax.invert_yaxis()

for i, s in enumerate(scores):
    ax.text(s + 0.02, i, f'{s:.2f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('relevance_scores.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.2 Source Diversity

In [ ]:
papers = result['raw_papers']
arxiv_count = sum(1 for p in papers if 'arxiv' in p.get('url', '').lower())
ss_count = len(papers) - arxiv_count

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['arXiv', 'Semantic Scholar'], [arxiv_count, ss_count], color=['#3498db', '#e74c3c'])
ax.set_ylabel('Number of Papers')
ax.set_title('Papers by Source')

for i, v in enumerate([arxiv_count, ss_count]):
    ax.text(i, v + 0.3, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('source_diversity.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"arXiv: {arxiv_count}, Semantic Scholar: {ss_count}")

### 7.3 Reading Order Visualization

In [ ]:
reading_order = result.get('reading_order', [])

if reading_order:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    positions = [entry.get('position', i+1) for i, entry in enumerate(reading_order)]
    short_titles = [entry.get('title', '')[:45] + '...' if len(entry.get('title', '')) > 45 
                    else entry.get('title', '') for entry in reading_order]
    
    colors = plt.cm.viridis([i / len(reading_order) for i in range(len(reading_order))])
    
    ax.barh(range(len(positions)), positions, color=colors)
    ax.set_yticks(range(len(short_titles)))
    ax.set_yticklabels(short_titles, fontsize=7)
    ax.set_xlabel('Reading Position')
    ax.set_title('Recommended Reading Order (1 = read first)')
    ax.invert_yaxis()
    
    plt.tight_layout()
    plt.savefig('reading_order.png', dpi=150, bbox_inches='tight')
    plt.show()

### 7.4 Papers by Publication Year

In [ ]:
years = [p.get('year', 0) for p in result['screened_papers'] if p.get('year')]
year_counts = {}
for y in years:
    year_counts[y] = year_counts.get(y, 0) + 1

sorted_years = sorted(year_counts.keys())
counts = [year_counts[y] for y in sorted_years]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([str(y) for y in sorted_years], counts, color='#9b59b6')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Papers')
ax.set_title('Screened Papers by Publication Year')

for i, v in enumerate(counts):
    ax.text(i, v + 0.1, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('papers_by_year.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Evaluation

Full evaluation suite: tool metrics, planning metrics, retrieval quality, efficiency, and baseline comparison. The baseline is a plain arXiv keyword search with no LLM involvement — this is what a student would get by just searching arXiv manually.

In [1]:
from evaluation.metrics import evaluate_run

eval_results = evaluate_run(result, elapsed, topic)


EVALUATION RESULTS

--- TOOL METRICS ---
  Tool Selection Accuracy: 1.00
  Tool Execution Success:  1.00 (4/4)
    search: PASS (arXiv + Semantic Scholar retrieval)
    screening: PASS (LLM relevance scoring)
    synthesis: PASS (LLM summarization + theme extraction)
    planning: PASS (LLM gap analysis + plan generation)

--- PLANNING METRICS ---
  Step Success Rate:    1.00 (7/7)
    Query Generation: PASS
    Paper Retrieval: PASS
    Relevance Screening: PASS
    Summarization: PASS
    Theme Identification: PASS
    Gap Analysis: PASS
    Research Plan: PASS
  Task Completion Rate: 1.00 (4/4)

--- RETRIEVAL & SCREENING METRICS ---
  Screening Precision:  1.00
  Coverage: 11/17 papers retained (0.65)
  Source Diversity:     arXiv=13, Semantic Scholar=4 (score: 0.308)
  Relevance Scores:    mean=0.773, min=0.6, max=0.9

--- EFFICIENCY METRICS ---
  Execution Time:      55.58s
  LLM API Calls:    ~4
  Iterations Used:     1
  Time Per Paper:      3.27s

--- BASELINE COMPARISON (Agen

### 8.1 Agentic vs Baseline Comparison

In [ ]:
categories = ['Papers Found', 'Sources Used', 'LLM Queries', 'Summaries', 'Themes', 'Gaps Found']
agentic_vals = [
    len(result['raw_papers']), 2, len(result['search_queries']),
    len(result['summaries']), len(result['themes']), len(result['research_gaps'])
]
baseline_vals = [10, 1, 1, 0, 0, 0]

x = range(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar([i - width/2 for i in x], agentic_vals, width, label='Agentic System', color='#2ecc71')
bars2 = ax.bar([i + width/2 for i in x], baseline_vals, width, label='Baseline (keyword search)', color='#e74c3c')

ax.set_ylabel('Count')
ax.set_title('Agentic System vs Non-Agentic Baseline')
ax.set_xticks(x)
ax.set_xticklabels(categories, rotation=15)
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
            str(int(bar.get_height())), ha='center', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
            str(int(bar.get_height())), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('agentic_vs_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Second Run — Different Topic

Running with a different topic to show the system generalizes.

In [ ]:
topic2 = "reinforcement learning from human feedback"

print(f"Running literature review for: {topic2}\n")

start2 = time.time()
result2 = run_literature_review(topic2, max_iterations=2)
elapsed2 = time.time() - start2

print(f"\nDone in {elapsed2:.1f}s")
print(f"Papers found: {len(result2['raw_papers'])}")
print(f"After screening: {len(result2['screened_papers'])}")
print(f"Themes: {result2['themes']}")
print(f"Gaps: {len(result2['research_gaps'])}")
print(f"Reading order: {len(result2.get('reading_order', []))} papers sequenced")

In [ ]:
# reading order for the second topic
reading_order2 = result2.get('reading_order', [])

if reading_order2:
    print(f"Reading path for '{topic2}':\n")
    for entry in reading_order2:
        pos = entry.get('position', '?')
        title = entry.get('title', 'Unknown')
        reason = entry.get('reason', '')
        print(f"  [{pos}] {title}")
        print(f"      -> {reason}")
        print()

In [ ]:
eval_results2 = evaluate_run(result2, elapsed2, topic2)

## 10. Technology Stack

| Component | Technology | Why |
|-----------|-----------|-----|
| Agent Orchestration | LangGraph | Stateful cyclic graphs, conditional edges for adaptive behavior |
| LLM | Groq (Llama 3.3 70B) | Free tier, generous limits, fast inference |
| Paper Retrieval | arXiv API + Semantic Scholar API | Two sources for better coverage |
| Framework | LangChain v0.3+ | Prompt templates, chain composition |
| State Management | Python TypedDict | Type-safe, works natively with LangGraph |
| Environment | Python 3.11, virtualenv | Standard setup |

We started with Google Gemini but ran into daily quota exhaustion during testing. The architecture is provider-agnostic — the LLM is initialized inside each agent function, so switching to Groq took about 5 minutes.

## 11. Summary

The system works end-to-end: given a research topic, it searches for papers across multiple sources, screens them for relevance, synthesizes themes, identifies gaps, and produces a **recommended reading order** for someone new to the topic.

**What makes it agentic:**
- LLM-driven query generation (not hardcoded keywords)
- Adaptive feedback loop — re-searches if not enough relevant papers
- LLM-based screening with numeric relevance scoring
- Cross-paper synthesis (themes emerge from the collection, not individual papers)
- Reading order reasoning based on concept dependencies

**Limitations:**
- Semantic Scholar has aggressive rate limits (429s are common)
- Reading order relies on LLM judgment, no citation graph analysis yet
- Relevance scores are LLM-generated, not validated against human judgments

**Possible extensions:**
- Citation graph analysis for more robust reading order
- PDF download + full-text analysis instead of abstract-only
- Interactive UI (Streamlit/Gradio) for non-technical users